In [0]:
import yaml
from pyspark.sql import functions as F

# Load project config
config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Catalog and schemas
catalog = config["catalog"]              # e.g., vattenfall_dev
silver_schema = "refined"
gold_schema = "gold"                     # target Gold schema

# Fully qualified table names
silver_market_prices = f"{catalog}.{silver_schema}.silver_market_prices"
gold_daily_market_summary = f"{catalog}.{gold_schema}.gold_daily_market_summary"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}")
print(f"Gold schema ready: {catalog}.{gold_schema}")

In [0]:
def log_table_info(source, target, df_source, df_target, aggregation_grain, metric_name):
    print(f"Source Table: {source}")
    print(f"Target Table: {target}")
    print(f"Rows Read: {df_source.count()}")
    print(f"Rows Written: {df_target.count()}")
    print(f"Aggregation Grain: {aggregation_grain}")
    print(f"Business Metric: {metric_name}")

In [0]:
def add_business_indicators(df):
    # High market price indicator (>100 EUR/MWh as example)
    df = df.withColumn("high_price_flag", F.when(F.col("price_eur_mwh") > 100, 1).otherwise(0))
    return df

In [0]:
# Read Silver table
df_silver = spark.table(silver_market_prices)

# Add business indicators
df_silver = add_business_indicators(df_silver)

# Define aggregation grain
grain = ["report_day", "region"]
metric_name = "daily_market_summary_metrics"

# Aggregate
df_gold = df_silver.groupBy(*grain).agg(
    F.avg("price_eur_mwh").alias("avg_market_price"),
    F.max("price_eur_mwh").alias("max_market_price"),
    F.sum("volume_mwh").alias("total_market_volume"),
    F.sum("high_price_flag").alias("high_price_count")
)

# Write to Gold
df_gold.write.mode("overwrite").saveAsTable(gold_daily_market_summary)

# Log summary
log_table_info(silver_market_prices, gold_daily_market_summary, df_silver, df_gold, grain, metric_name)

# Preview


In [0]:
df_gold.show(5)